# 🎬 Subtitle ML Pipeline — Notebook
**Features:** Video → SRT | Audio → SRT | Text → SRT | Dataset Builder | Whisper Fine-Tuning

## 0. Install Dependencies

In [1]:
!pip install openai-whisper moviepy torch torchaudio datasets\
    transformers pydub tqdm rich evaluate jiwer accelerate

  Using cached moviepy-2.2.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached jiwer-4.0.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached imageio-2.37.2-py3-none-any.whl.metadata (9.7 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-macosx_10_9_intel.macosx_10_9_x86_64.whl.metadata (1.5 kB)
  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_x86_64.whl.metadata (62 kB)
  Using cached proglog-0.1.12-py3-none-any.whl.metadata (794 bytes)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
INFO: pip is looking at multiple versions of numba to determine which version is compatible with other requirements. This could take a while.
  Using cached numba-0.64.0-cp310-cp310-macosx_14_0_x86_64.whl
  Using cached llvmlite-0.46.0.tar.gz (193 kB)
  Installing build dependencies ... done
  Ge

In [2]:
import sys
sys.path.insert(0, '.')
from subtitle_pipeline import (
    SubtitleMLPipeline, TextToSRT, DatasetBuilder,
    WhisperFineTuner, SubtitleWriter
)
import torch
print('GPU available:', torch.cuda.is_available())

ModuleNotFoundError: No module named 'subtitle_pipeline'

## 1. 🎥 Video → Subtitles

In [ ]:
import whisper
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datetime import timedelta


class SubtitleMLPipeline:
    def __init__(self):

        # Device (Mac M1/M2 supports MPS)
        if torch.backends.mps.is_available():
            self.device = "mps"
        else:
            self.device = "cpu"

        # Lazy load models (DON'T load heavy models at startup)
        self.whisper_model = None
        self.tokenizer = None
        self.translator = None

        self.translator_name = "facebook/nllb-200-distilled-600M"

    # -----------------------------
    # Load Whisper Only When Needed
    # -----------------------------
    def load_whisper(self):
        if self.whisper_model is None:
            self.whisper_model = whisper.load_model("base")
        return self.whisper_model

    # -----------------------------
    # Load Translator Only When Needed
    # -----------------------------
    def load_translator(self):
        if self.tokenizer is None or self.translator is None:
            self.tokenizer = AutoTokenizer.from_pretrained(self.translator_name)
            self.translator = AutoModelForSeq2SeqLM.from_pretrained(
                self.translator_name
            ).to(self.device)

    # -----------------------------
    # Timestamp Formatter
    # -----------------------------
    def format_timestamp(self, seconds):
        millis = int((seconds - int(seconds)) * 1000)
        td = timedelta(seconds=int(seconds))
        formatted = str(td)
        if "." not in formatted:
            formatted += ".000000"
        formatted = formatted.split(".")[0]
        return formatted.replace(".", ":") + f",{millis:03d}"

    # -----------------------------
    # Transcribe Audio
    # -----------------------------
    def transcribe(self, audio_path):
        model = self.load_whisper()
        result = model.transcribe(audio_path)
        return result["segments"], result.get("language")

    # -----------------------------
    # Translate Text
    # -----------------------------
    def translate_text(self, text, src_lang, tgt_lang):
        self.load_translator()

        self.tokenizer.src_lang = src_lang
        encoded = self.tokenizer(text, return_tensors="pt").to(self.device)

        generated = self.translator.generate(
            **encoded,
            forced_bos_token_id=self.tokenizer.lang_code_to_id[tgt_lang],
            max_length=512
        )

        return self.tokenizer.batch_decode(
            generated,
            skip_special_tokens=True
        )[0]

    # -----------------------------
    # Audio → SRT Pipeline
    # -----------------------------
    def audio_to_srt(self, audio_path, out_path,
                     src_lang=None, tgt_lang=None):

        segments, detected_lang = self.transcribe(audio_path)

        if not src_lang:
            src_lang = detected_lang + "_Latn"

        with open(out_path, "w", encoding="utf-8") as f:
            for i, seg in enumerate(segments):

                text = seg["text"].strip()

                if tgt_lang and tgt_lang != src_lang:
                    text = self.translate_text(text, src_lang, tgt_lang)

                start = self.format_timestamp(seg["start"])
                end = self.format_timestamp(seg["end"])

                f.write(f"{i+1}\n")
                f.write(f"{start} --> {end}\n")
                f.write(text + "\n\n")

    # -----------------------------
    # Text → SRT
    # -----------------------------
    def text_to_srt(self, text, out_path):
        with open(out_path, "w", encoding="utf-8") as f:
            f.write("1\n")
            f.write("00:00:00,000 --> 00:00:10,000\n")
            f.write(text.strip())

## 2. 🎙️ Voice / Audio → Subtitles

In [ ]:
pipeline.audio_to_srt(
    audio_path='recording.wav',
    srt_out='output/voice_subtitles.srt',
    language='en',
)

# Preview result
with open('output/voice_subtitles.srt') as f:
    print(f.read()[:1000])

## 3. 📝 Text → Auto Subtitles (.srt)

In [ ]:
# From plain text string
pipeline.text_to_srt(
    text='Hello and welcome to this tutorial. Today we will learn about machine learning. '
         'Subtitles are automatically generated from plain text.',
    srt_out='output/text_subtitles.srt',
    words_per_subtitle=8,
)

# From a .txt file
import pathlib
txt = pathlib.Path('my_script.txt').read_text()
pipeline.text_to_srt(txt, 'output/script_subtitles.srt')

# From timestamped JSON
from subtitle_pipeline import TextToSRT, SubtitleWriter
converter = TextToSRT()
writer = SubtitleWriter()

json_data = [
    {'start': 0.0,  'end': 3.0,  'text': 'Welcome to the demo.'},
    {'start': 3.5,  'end': 7.0,  'text': 'This is a custom subtitle.'},
    {'start': 7.5,  'end': 11.0, 'text': 'Generated from JSON timestamps.'},
]
result = converter.from_json(json_data)
writer.save_srt(result, 'output/json_subtitles.srt')
print(result.to_srt())

## 4. 🗂️ Dataset Builder

In [ ]:
# Slice audio into segments aligned with SRT timestamps
records = pipeline.build_dataset(
    srt_path='output/video_subtitles.srt',
    audio_path='data/video_audio.wav',
    out_dir='data/dataset',
)

import pandas as pd
df = pd.DataFrame(records)
print(df.head())
print(f'\nTotal segments: {len(df)}')
print(f'Total duration: {df.duration.sum():.1f}s')

## 5. 🤖 Fine-Tune Whisper on Custom Dataset

In [ ]:
# Fine-tune Whisper-small on your dataset
pipeline.fine_tune(
    manifest_path='data/dataset/manifest.json',
    base_model='openai/whisper-small',
    language='english',       # change to your target language
    epochs=3,
    output_dir='models/whisper-finetuned',
)

## 6. 🔁 Use Fine-Tuned Model for Inference

In [ ]:
from transformers import pipeline as hf_pipeline

asr = hf_pipeline(
    'automatic-speech-recognition',
    model='models/whisper-finetuned',
    chunk_length_s=30,
    return_timestamps=True,
)

out = asr('recording.wav')
for chunk in out['chunks']:
    print(f"{chunk['timestamp']} → {chunk['text']}")

/Users/basanirajrao/Desktop/subtitle_ml_project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


OSError: models/whisper-finetuned is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

## 7. 📊 Evaluate WER on Test Set

In [ ]:
import evaluate, json
wer = evaluate.load('wer')

# Load test predictions vs references
references = ['hello world', 'this is a test']
predictions = ['hello world', 'this is a test']

score = wer.compute(predictions=predictions, references=references)
print(f'WER: {score:.2%}')